# UC-JOINS-1 — Named Collection Join via OGC API - Joins

**Persona:** Data Analyst

**Goal:** Create two collections in the same catalog (a primary with spatial
features and a secondary with tabular attributes), then issue a
`POST /join/catalogs/{cat}/collections/{col}/join` to enrich the primary
features with attributes from the secondary collection, joining on a shared
key column.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `joins` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)

**References:**
- OGC API - Joins Part 1 (working draft)
- Routes: `GET /join/catalogs/{cat}/collections/{col}/join` (describe),
  `POST /join/catalogs/{cat}/collections/{col}/join` (execute)

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-joins")
PRIMARY_COL   = os.environ.get("PRIMARY_COLLECTION", "admin-regions")
SECONDARY_COL = os.environ.get("SECONDARY_COLLECTION", "region-stats")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform         : {BASE_URL}")
print(f"Catalog          : {CATALOG_ID}")
print(f"Primary collection : {PRIMARY_COL}")
print(f"Secondary collection: {SECONDARY_COL}")

## Step 1 — Create catalog and ingest primary features

The primary collection holds spatial features (polygon regions) that will be
the left side of the join. Each feature carries a `region_code` property that
acts as the join key.

In [ ]:
# --- catalog ---
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "Joins demo catalog"}),
)
print(f"catalog   : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- primary collection ---
r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({"id": PRIMARY_COL, "title": "Administrative regions (primary)"}),
)
print(f"primary   : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- primary features ---
primary_features = [
    {"id": "lazio",   "region_code": "IT-LAZ", "coords": [[11.5,41.2],[13.9,41.2],[13.9,42.9],[11.5,42.9],[11.5,41.2]]},
    {"id": "toscana", "region_code": "IT-TOS", "coords": [[9.7,42.4],[12.4,42.4],[12.4,44.5],[9.7,44.5],[9.7,42.4]]},
    {"id": "umbria",  "region_code": "IT-UMB", "coords": [[11.9,42.3],[13.3,42.3],[13.3,43.7],[11.9,43.7],[11.9,42.3]]},
]

for feat_def in primary_features:
    feat = {
        "type": "Feature",
        "id": feat_def["id"],
        "geometry": {"type": "Polygon", "coordinates": [feat_def["coords"]]},
        "properties": {"region_code": feat_def["region_code"]},
    }
    r = client.post(
        f"/features/catalogs/{CATALOG_ID}/collections/{PRIMARY_COL}/items",
        content=json.dumps(feat),
    )
    if r.status_code in (200, 201):
        print(f"  primary item '{feat_def['id']}': created")
    elif r.status_code == 409:
        print(f"  primary item '{feat_def['id']}': already exists")
    else:
        print(f"  primary item '{feat_def['id']}': {r.status_code} {r.text[:200]}")

## Step 2 — Ingest secondary (attribute) features

The secondary collection holds tabular attribute records identified by the
same `region_code` join key. These features have no geometry (null) — the
joins extension accepts null-geometry features as lookup rows.

In [ ]:
# --- secondary collection ---
r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({"id": SECONDARY_COL, "title": "Region statistics (secondary)"}),
)
print(f"secondary : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- secondary records ---
secondary_records = [
    {"id": "stats-laz", "region_code": "IT-LAZ", "population": 5879082,  "area_km2": 17232},
    {"id": "stats-tos", "region_code": "IT-TOS", "population": 3692555,  "area_km2": 22987},
    {"id": "stats-umb", "region_code": "IT-UMB", "population":  882015,  "area_km2":  8464},
]

for rec in secondary_records:
    feat = {
        "type": "Feature",
        "id": rec["id"],
        "geometry": None,
        "properties": {
            "region_code": rec["region_code"],
            "population":  rec["population"],
            "area_km2":    rec["area_km2"],
        },
    }
    r = client.post(
        f"/features/catalogs/{CATALOG_ID}/collections/{SECONDARY_COL}/items",
        content=json.dumps(feat),
    )
    if r.status_code in (200, 201):
        print(f"  secondary item '{rec['id']}': created")
    elif r.status_code == 409:
        print(f"  secondary item '{rec['id']}': already exists")
    else:
        print(f"  secondary item '{rec['id']}': {r.status_code} {r.text[:200]}")

## Step 3 — Describe join capabilities

`GET /join/catalogs/{cat}/collections/{col}/join` returns the supported
secondary driver types for this collection. A 200 response confirms the
joins extension is active.

In [ ]:
r = client.get(f"/join/catalogs/{CATALOG_ID}/collections/{PRIMARY_COL}/join")
print(f"join describe status: {r.status_code}")

_joins_active = r.status_code == 200

if _joins_active:
    desc = r.json()
    print(f"  title   : {desc.get('title')}")
    print(f"  primary : {desc.get('primary')}")
    print(f"  supported_secondary_drivers: {desc.get('supported_secondary_drivers')}")
elif r.status_code == 404:
    print("SKIP: /join endpoint not found — joins extension may not be active.")
else:
    print(r.text[:300])

## Step 4 — Execute a named collection join

`POST /join/catalogs/{cat}/collections/{col}/join` with a `NamedSecondarySpec`
body joins the primary collection features with the secondary collection on
the `region_code` key. The response is a GeoJSON FeatureCollection where each
primary feature has the secondary properties merged into its `properties`.

In [ ]:
if not _joins_active:
    print("SKIP: joins extension not active.")
else:
    join_payload = {
        "secondary": {
            "type": "registered",
            "ref": SECONDARY_COL,
        },
        "join": {
            "primary_column": "region_code",
            "secondary_column": "region_code",
            "mode": "inner",
        },
        "paging": {"limit": 100},
    }

    r = client.post(
        f"/join/catalogs/{CATALOG_ID}/collections/{PRIMARY_COL}/join",
        content=json.dumps(join_payload),
    )
    print(f"join execute status: {r.status_code}")

    if r.status_code == 200:
        result = r.json()
        features = result.get("features", [])
        meta = result.get("_join_meta", {})
        print(f"  features joined : {len(features)}")
        print(f"  join_meta       : {meta}")
        for feat in features:
            props = feat.get("properties", {})
            print(
                f"  id={feat.get('id')}  region_code={props.get('region_code')}  "
                f"population={props.get('population')}  area_km2={props.get('area_km2')}"
            )
    elif r.status_code == 404:
        print("  404 — catalog, collection or join driver not found.")
    elif r.status_code == 400:
        print(f"  400 — {r.text[:300]}")
    else:
        print(r.text[:300])

## Teardown

Remove the demo collections and catalog.

In [ ]:
_TEARDOWN = os.environ.get("JOINS_TEARDOWN", "true").lower() == "true"

if _TEARDOWN:
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{SECONDARY_COL}")
    print(f"DELETE secondary : {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{PRIMARY_COL}")
    print(f"DELETE primary   : {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog   : {r.status_code}")
else:
    print("Teardown skipped (set JOINS_TEARDOWN=true to enable).")

## Result — OGC Joins landing URL

The joins extension does not ship a dedicated web page. The canonical OGC
entry point is the join-describe endpoint below.

In [ ]:
client.close()
print(
    f"Join describe URL: "
    f"{BASE_URL}/join/catalogs/{CATALOG_ID}/collections/{PRIMARY_COL}/join"
)